# Subband coherence across the cochleogram

The duration-weighted grand-average coherence, resolved over the **60 gammatone
subbands** rather than collapsed to the broadband envelope. Where figures 2-4 ask at
what *modulation* frequency the EEG follows the speech, this one adds the second axis:
which part of the **cochlear** spectrum that modulation is carried in.

Ported from `exploration/subband_coherence.ipynb` -- its final figure. Left there: the
cross-check against the broadband pipeline, and the intermediate figure variants that
led to this layout.

## Figure produced

**`subband_coherence_<reference>`** -- two panels sharing a cochlear centre-frequency
axis, both averaged over the whole scalp:

* **(a)** 0.1-30 Hz, unsmoothed, on a continuous scale.
* **(b)** 30-300 Hz, mains harmonics dropped and the result smoothed along frequency,
  on a discrete scale, with the pooled F1-F3 ranges marked.

## Input

One file: `<coherence_dir>/reference_average/average_coherence_<reference>.npz`, from
`scripts/coherence/get_referenced_coherence.py`, which computes every reference in one
~25 min pass. `get_null_cochleogram_coherence.py` produces the same array to 8.8e-08
but alongside a 500-draw subband null costing ~40 GB and ~10 h per reference -- and
nothing here reads that null, so it is not worth running for this figure.

## Setup

`REFERENCE` selects the EEG reference; the input file and the figure name follow from
it. The channel axis is the reference's own, not the raw file's:

| `REFERENCE` | sensors | trials |
|---|---|---|
| `"avg"` | 64 | 563 |
| `"linked_mastoids"` | 64 | 542; 21 recordings carry no A1/A2 and are skipped |
| `"FCz"` | 63 | 563; FCz is identically zero once referenced to itself, and is dropped |

Every sensor average and every inset is taken from the **file's own** `channel_names`,
so a reference that carries a different set cannot be averaged against the wrong one.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm
from mne import pick_info
from mne.io import read_raw_fif
from scipy.ndimage import uniform_filter1d

ROOT = Path.cwd().parent if Path.cwd().name == "figures" else Path.cwd()
CONFIG = json.loads((ROOT / "src" / "package_data" / "config.json").read_text())

REFERENCE = "linked_mastoids"  # "avg" | "linked_mastoids" | "FCz"

sys.path.insert(0, str(ROOT / "src"))

OUT_DIR = ROOT / "figures"
# One directory holds every reference: the files are named by reference, not filed
# under it.
REFERENCE_AVERAGE = (Path(CONFIG["coherence_dir"]) / "reference_average"
                     / f"average_coherence_{REFERENCE}.npz")
FORMANT_DIR = Path(CONFIG["speech_units_dir"]) / "formants"
EEG_DIR = Path(CONFIG["eeg_dir"])
N_SUBBANDS = 60

# Sensor positions. The pipeline builds its EEG through pipeline.load_eeg, which does
# raw.pick("eeg"), so the saved channel axis is a subset of this info's order in the
# same order. It is narrowed to the file's own channels once that file is read.
TEMPLATE_EEG = EEG_DIR / "sub-001_-_ses-001_-_trial-001_-_cond-natural_-_audio01-audiobook_5_1_-_eeg.fif"
RAW_INFO = read_raw_fif(TEMPLATE_EEG, preload=False, verbose="ERROR").pick("eeg").info

# --- house style, shared with figures 1-4 ------------------------------------
DEFAULT_FONTSIZE = 10
TICK_FONTSIZE = 8
PANEL_LABEL_FONTSIZE = 14

INK = "#1b1b1a"

plt.rcParams.update({
    "font.size": TICK_FONTSIZE,
    "axes.labelsize": DEFAULT_FONTSIZE,
    "axes.titlesize": DEFAULT_FONTSIZE,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "xtick.labelsize": TICK_FONTSIZE,
    "ytick.labelsize": TICK_FONTSIZE,
    "axes.linewidth": 0.8,
    "figure.dpi": 130,
    # TrueType, not matplotlib's default Type 3, which publishers flag.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def bare(ax):
    """Recessive chrome: drop the top/right rules."""
    ax.spines[["top", "right"]].set_visible(False)
    return ax


print(f"reference: {REFERENCE}  ({REFERENCE_AVERAGE.name}, "
      f"{REFERENCE_AVERAGE.stat().st_size / 1e6:.0f} MB)")

## Loading

`np.load` on an `.npz` decompresses a member when it is indexed, not when the file is
opened, so only what the figure needs is read.

Listening time per stimulus comes from this file's own per-trial durations, so the
figure does not depend on a 500-draw null run having happened for this reference.

In [ ]:
with np.load(REFERENCE_AVERAGE) as saved:
    if "subband_coherence" not in saved.files:
        raise KeyError(f"{REFERENCE_AVERAGE.name} predates the subband arrays; re-run "
                       "scripts/coherence/get_referenced_coherence.py")
    subband_coherence = saved["subband_coherence"]                 # (F, 60, n_eeg)
    centre_frequencies = saved["centre_frequencies"]
    frequencies = saved["frequencies"]
    channel_names = [str(name) for name in saved["channel_names"]]
    trial_stems = [str(stem) for stem in saved["trial_stems"]]
    trial_durations = saved["trial_durations"]
    n_trials = int(saved["n_trials"])
    n_skipped = int(saved["skipped_stems"].shape[0])
    hours = float(saved["total_duration"]) / 3600

assert subband_coherence.shape[1] == N_SUBBANDS, f"expected {N_SUBBANDS} subbands"
assert np.all(np.diff(centre_frequencies) > 0), "subbands not in ascending centre order"

# INFO narrowed to the channels this file actually carries, in its order -- so every
# sensor average and every inset indexes the same channels the data has.
INFO = pick_info(RAW_INFO, [RAW_INFO.ch_names.index(ch) for ch in channel_names])
assert list(INFO.ch_names) == channel_names, "channel axis differs from the file's"

# Listening time per stimulus, for pooling the formant histograms below.
stimulus_seconds = {}
for stem, duration in zip(trial_stems, trial_durations):
    audio_name = stem.split("_-_")[4].removeprefix("audio01-")
    stimulus_seconds[audio_name] = stimulus_seconds.get(audio_name, 0.0) + float(duration)

power = np.abs(subband_coherence) ** 2
skipped = f", {n_skipped} skipped (no A1/A2)" if n_skipped else ""
print(f"{REFERENCE}: {N_SUBBANDS} subbands, {centre_frequencies[0]:.0f}-"
      f"{centre_frequencies[-1]:.0f} Hz; {n_trials} trials{skipped}, {hours:.1f} h, "
      f"{len(channel_names)} sensors; {len(frequencies)} bins at "
      f"{frequencies[1] - frequencies[0]:.1f} Hz")
print(f"{len(stimulus_seconds)} stimuli, for the formant pooling below")

## Formant ranges

Each stimulus's F1-F3 histogram (10 Hz bins) is weighted by its listening time and
summed, and a formant's range is the central 99% of that pooled distribution.
Weighting each stimulus's histogram as a density instead moves no edge by more than
10 Hz.

In [ ]:
FORMANT_NAMES = ["F1", "F2", "F3"]
FORMANT_COVERAGE = 0.99

pooled_formants = 0.0
for name, seconds in stimulus_seconds.items():
    with np.load(FORMANT_DIR / f"{name}_formants.npz") as saved:
        pooled_formants = (pooled_formants
                           + np.nan_to_num(saved["counts"][:3].astype(float)) * seconds)
        formant_bins = saved["bin_centres"]

cdf = pooled_formants.cumsum(1) / pooled_formants.sum(1, keepdims=True)
tail = (1 - FORMANT_COVERAGE) / 2
formant_low = formant_bins[np.argmax(cdf >= tail, axis=1)]
formant_high = formant_bins[np.argmax(cdf >= 1 - tail, axis=1)]
for label, low, high in zip(FORMANT_NAMES, formant_low, formant_high):
    print(f"{label}: {low:.0f}-{high:.0f} Hz")

## Panels

One function per regime, and one for the sensor inset they share.

The two regimes are drawn differently because they carry different amounts of signal.
Panel (a) is unsmoothed on a continuous autoscaled colour bar. Panel (b) drops the
mains harmonics, smooths along frequency over 6 Hz, and uses a **discrete** scale of
nine fixed steps -- fixed, so the panel is comparable between references rather than
rescaled to whichever one is being drawn.

`PANEL_B_SENSORS` chooses what panel (b) averages. The default is the whole scalp, the
same set panel (a) uses, so a difference between the panels cannot come from the choice
of sensors. `CENTRAL_CLUSTER` is the alternative -- the midline strip where
high-frequency tracking is focal in figure 2's topographies.

In [ ]:
# The central cluster, kept as the alternative for PANEL_B_SENSORS below.
CENTRAL_CLUSTER = [
    "F1", "Fz", "F2",
    "FC1", "FCz", "FC2",
    "C1", "Cz", "C2",
    "CP1", "CPz", "CP2",
    "P1", "Pz", "P2",
]

# What each panel averages. Intersected with what this reference actually carries --
# FCz is in the central cluster and is dropped by its own reference -- so a missing
# sensor is reported rather than raising or silently averaging a different set.
ALL_SENSORS = list(INFO.ch_names)
PANEL_B_SENSORS = ALL_SENSORS          # or CENTRAL_CLUSTER


def available(sensors, label):
    kept = [ch for ch in sensors if ch in channel_names]
    missing = sorted(set(sensors) - set(kept))
    if missing:
        print(f"{label}: {len(kept)} of {len(sensors)} sensors "
              f"({', '.join(missing)} not in the {REFERENCE} reference)")
    return kept


panel_a_sensors = available(ALL_SENSORS, "panel (a)")
panel_b_sensors = available(PANEL_B_SENSORS, "panel (b)")

# Mains hum and its harmonics are a shared artefact of both signals -- figure 2's
# definition, so a bin dropped here is a bin dropped there.
LINE_FREQ, NOTCH_HALFWIDTH = 50.0, 2.0
# Panel (b)'s smoothing along frequency: 30 bins, i.e. 6 Hz.
SMOOTHING_BINS = 25
# Both panels use a discrete scale. Fixed rather than autoscaled, so a panel is
# comparable between references rather than rescaled to whichever one is being drawn,
# and banded rather than continuous so a level can be read off the plate instead of
# being estimated against a gradient.
#
# The two ranges differ by about a factor of four -- the low regime carries that much
# more coherence -- so they cannot share one ladder without flattening panel (b)
# entirely.
#
# Panel (a): 0.005 steps to 0.045, every edge labelled. Its top bound sits above the
# maximum the data reaches (0.0423), so the extension block is empty under this
# reference and is kept only in case another exceeds it.
LOW_STEP = 0.005
LOW_BOUNDS = np.arange(0.0, 0.0451, LOW_STEP)
HIGH_BOUNDS = np.linspace(0.00, 0.01, 10)

# One colormap for both panels. The ladders differ, so a colour does not mean the same
# value in the two -- what it does mean is that a reader is reading one scale's worth
# of colour, and a band's position in the ramp is comparable between them.
CMAP = "plasma"

CENTRE_TICKS = [50, 100, 1000, 8000]
# Formant brackets: x position, and the height of each label. Placed in the empty top
# right of panel (b).
FORMANT_X = [240, 255, 270]
FORMANT_LABEL_Y = [500, 1400, 2700]


def rms_over(sensors):
    """RMS coherence magnitude over the named sensors, per frequency and subband."""
    index = [INFO.ch_names.index(ch) for ch in sensors]
    return np.sqrt(power[..., index].mean(-1))                          # (F, 60)


def nan_boxcar(values, size):
    """Running mean along frequency that skips NaNs.

    The mains bins are set to NaN before smoothing rather than after: a boxcar over a
    line artefact would spread it into its neighbours, and removing the bins
    afterwards would leave that contamination just outside the notch. Dropped bins are
    filled from their neighbours instead.
    """
    valid = ~np.isnan(values)
    total = uniform_filter1d(np.where(valid, values, 0.0), size, axis=0, mode="nearest")
    count = uniform_filter1d(valid.astype(float), size, axis=0, mode="nearest")
    return total / count


def banded_colorbar(mesh, ax, bounds, label_every=1):
    """A colourbar with a square extension cap, labelled every `label_every` edge.

    `extendrect` squares off the over-range block; the default draws it as a triangle,
    which reads as an arrow pointing out of the scale rather than as one more band.

    `label_every` separates how fine the bands are from how many of them get a number.
    A bar this short cannot carry a label per edge once the step is small, and the
    alternative -- a coarser step -- costs the plate its resolution rather than the
    bar its clutter.
    """
    colorbar = plt.colorbar(mesh, ax=ax, pad=0.02, extendrect=True)
    ticks = bounds[::label_every]
    colorbar.set_ticks(ticks, labels=[f"{tick:.3f}" for tick in ticks])
    colorbar.ax.tick_params(labelsize=TICK_FONTSIZE)
    return colorbar


def centre_frequency_axis(ax, label=True):
    ax.set_yscale("log")
    ax.set_yticks(CENTRE_TICKS, CENTRE_TICKS if label else [""] * len(CENTRE_TICKS))
    if label:
        ax.set_ylabel("Cochlear centre frequency (Hz)")
    ax.set_xlabel("Frequency for coherence (Hz)")
    bare(ax)


def plot_low_modulation(ax, sensors):
    """Panel (a): 0.1-30 Hz, unsmoothed, discrete scale."""
    norm = BoundaryNorm(LOW_BOUNDS, len(LOW_BOUNDS), extend="max")
    mesh = ax.pcolormesh(frequencies[1:], centre_frequencies, rms_over(sensors)[1:].T,
                         shading="auto", cmap=plt.get_cmap(CMAP, len(LOW_BOUNDS)),
                         norm=norm, rasterized=True)
    ax.set_xlim(0.1, 30)
    ax.set_xticks(range(5, 31, 5), range(5, 31, 5))
    centre_frequency_axis(ax)
    banded_colorbar(mesh, ax, LOW_BOUNDS)


def plot_high_modulation(ax, sensors):
    """Panel (b): 30-300 Hz, mains dropped, smoothed, discrete scale."""
    values = rms_over(sensors)
    harmonics = np.arange(LINE_FREQ, frequencies[-1] + LINE_FREQ, LINE_FREQ)
    line = (np.abs(frequencies[:, None] - harmonics) <= NOTCH_HALFWIDTH).any(1)
    values[line] = np.nan
    values = nan_boxcar(values, SMOOTHING_BINS)

    norm = BoundaryNorm(HIGH_BOUNDS, len(HIGH_BOUNDS), extend="max")
    mesh = ax.pcolormesh(frequencies[1:], centre_frequencies, values[1:].T,
                         shading="auto", cmap=plt.get_cmap(CMAP, len(HIGH_BOUNDS)),
                         norm=norm, rasterized=True)
    ax.set_xlim(20, 300)
    ax.set_xticks(range(50, 301, 50), range(50, 301, 50))
    # The shared cochlear axis is labelled once, on panel (a).
    centre_frequency_axis(ax, label=False)

    for x, low, high, name, label_y in zip(FORMANT_X, formant_low, formant_high,
                                           FORMANT_NAMES, FORMANT_LABEL_Y):
        ax.plot([x, x], [low, high], c="white", lw=2)
        ax.plot([x - 2, x + 2], [low, low], c="white", lw=2)
        ax.plot([x - 2, x + 2], [high, high], c="white", lw=2)
        ax.text(x + 15, label_y, name, color="white", fontsize=TICK_FONTSIZE,
                ha="center", weight="bold")

    banded_colorbar(mesh, ax, HIGH_BOUNDS)

## The figure

Both panels on one row at the 6.27 in text-block width the other figures use.

In [ ]:
FIGSIZE = (6.27, 2.6)
# The gutter has to hold panel (a)'s colourbar and its tick labels as well as separate
# the two plates -- a colourbar takes its space out of its own panel's box, not out of
# this gap -- so it is wider than a bare pair of axes would need.
PANEL_GAP = 0.22

fig, axs = plt.subplots(1, 2, figsize=FIGSIZE)
fig.subplots_adjust(wspace=PANEL_GAP)

plot_low_modulation(axs[0], panel_a_sensors)
plot_high_modulation(axs[1], panel_b_sensors)
axs[0].set_title("Low modulation frequencies")
axs[1].set_title("High modulation frequencies")

# No suptitle: the reference, trial count and listening time belong in the caption,
# as in figures 3 and 4. The load cell prints them.

for ax, letter in zip(axs, "ab"):
    ax.text(-0.2, 1.05, f"{letter})", transform=ax.transAxes,
            fontsize=PANEL_LABEL_FONTSIZE, fontweight="bold", va="bottom", ha="left")

stem = f"subband_coherence_{REFERENCE}"
fig.savefig(OUT_DIR / f"{stem}.pdf", bbox_inches="tight", pad_inches=0.3)
fig.savefig(OUT_DIR / f"{stem}.png", bbox_inches="tight", pad_inches=0.3, dpi=200)
plt.show()